# 🗄️ Trabalho Prático — Python e SQLite

### Manipulação de Banco de Dados Relacional com `sqlite3`

---

**Disciplina:** Banco de Dados
**Portfólio:** Entrega individual
**Linguagem:** Python 3
**Banco de dados:** SQLite (`empresa.db`)

---

## 📋 Sobre este notebook

Este notebook documenta, passo a passo, a criação e manipulação de um banco de dados relacional (`empresa.db`) utilizando a biblioteca nativa `sqlite3` do Python. O trabalho está organizado em **6 partes**, seguindo exatamente a estrutura proposta no roteiro da disciplina:

| Parte | Conteúdo |
|:-----:|----------|
| 1 | Configuração e criação do banco de dados (DDL) |
| 2 | Manipulação de dados — INSERT, SELECT, UPDATE, DELETE (DML) |
| 3 | Agregação e ordenação (SUM, AVG, GROUP BY, ORDER BY) |
| 4 | Transações — COMMIT e ROLLBACK |
| 5 | Múltiplas tabelas e JOINs (INNER e LEFT) |
| 6 | Conceito de DCL |

Cada etapa é acompanhada de uma célula de explicação em Markdown, seguida da célula de código correspondente e, quando aplicável, uma breve análise do resultado obtido.


---
## 🧩 Parte 1 — Configuração e Criação do Banco de Dados

Nesta primeira etapa, o objetivo é **criar o arquivo de banco de dados** `empresa.db`, **estabelecer a conexão** com ele através da biblioteca `sqlite3` e **criar a tabela `funcionarios`**, que será a base de todo o restante do trabalho.

> 💡 **Observação:** o SQLite cria o arquivo `.db` automaticamente no momento em que a conexão é aberta, caso ele ainda não exista.


### 1.1 — Importação da biblioteca, criação do banco e verificação da conexão

In [ ]:
import sqlite3

# Cria (ou abre, caso já exista) o banco de dados "empresa.db"
conn = sqlite3.connect("empresa.db")

# O objeto cursor é quem efetivamente executa os comandos SQL
cursor = conn.cursor()

# Verificação simples de que a conexão foi estabelecida com sucesso
if conn:
    print("✅ Conexão com o banco de dados 'empresa.db' realizada com sucesso!")
else:
    print("❌ Falha ao conectar ao banco de dados.")


### 1.2 — Criação da tabela `funcionarios` (DDL)

A tabela `funcionarios` será criada com o comando `CREATE TABLE IF NOT EXISTS`, o que evita erros caso o notebook seja executado mais de uma vez. As colunas seguem exatamente o que foi solicitado:

- **id** → `INTEGER PRIMARY KEY AUTOINCREMENT`
- **nome** → `TEXT NOT NULL`
- **cargo** → `TEXT NOT NULL`
- **salario** → `REAL`
- **data_contratacao** → `TEXT NOT NULL`


In [ ]:
criar_tabela_funcionarios = """
CREATE TABLE IF NOT EXISTS funcionarios (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nome TEXT NOT NULL,
    cargo TEXT NOT NULL,
    salario REAL,
    data_contratacao TEXT NOT NULL
);
"""

cursor.execute(criar_tabela_funcionarios)
conn.commit()

print("✅ Tabela 'funcionarios' criada com sucesso!")


### 1.3 — Verificação da estrutura da tabela

Para confirmar se as colunas foram criadas corretamente, usamos o comando `PRAGMA table_info(nome_da_tabela)`, um recurso específico do SQLite que retorna metadados sobre a tabela (nome, tipo, se aceita nulo, etc.).


In [ ]:
cursor.execute("PRAGMA table_info(funcionarios);")
estrutura = cursor.fetchall()

print(f"{'cid':<4}{'nome':<20}{'tipo':<10}{'notnull':<9}{'default':<10}{'pk'}")
print("-" * 60)
for coluna in estrutura:
    cid, nome, tipo, notnull, default, pk = coluna
    print(f"{cid:<4}{nome:<20}{tipo:<10}{notnull:<9}{str(default):<10}{pk}")


---
## 🧩 Parte 2 — Manipulação de Dados (DML)

Aqui trabalhamos os quatro comandos clássicos de **DML (Data Manipulation Language)**: `INSERT`, `SELECT`, `UPDATE` e `DELETE`, além de um `SELECT COUNT(*)`.

> ⚠️ Utilizamos **placeholders (`?`)** nos comandos `INSERT` em vez de concatenar strings diretamente. Essa é a forma correta e segura de inserir dados no SQLite, pois evita problemas de formatação e ataques de *SQL Injection*.


### 2.1 — Inserção de 5 registros fictícios (INSERT)

In [ ]:
funcionarios_iniciais = [
    ("João Silva",     "Analista",      4500.00, "2020-03-15"),
    ("Maria Souza",    "Gerente",       7500.00, "2019-07-01"),
    ("Carlos Pereira", "Desenvolvedor", 6000.00, "2021-01-10"),
    ("Ana Lima",       "Gerente",       8200.00, "2018-11-23"),
    ("Pedro Santos",   "Analista",      4800.00, "2022-05-30"),
]

cursor.executemany(
    "INSERT INTO funcionarios (nome, cargo, salario, data_contratacao) VALUES (?, ?, ?, ?);",
    funcionarios_iniciais
)
conn.commit()

print(f"✅ {cursor.rowcount if cursor.rowcount != -1 else len(funcionarios_iniciais)} registros inseridos com sucesso!")


### 2.2 — Consulta de todos os registros (SELECT)

In [ ]:
cursor.execute("SELECT * FROM funcionarios;")
todos = cursor.fetchall()

print("Todos os funcionários cadastrados:\n")
for f in todos:
    print(f)


### 2.3 — Filtros com WHERE

In [ ]:
# Funcionários com salário superior a R$ 5.000,00
cursor.execute("SELECT * FROM funcionarios WHERE salario > 5000;")
salarios_altos = cursor.fetchall()

print("Funcionários com salário > R$ 5.000,00:\n")
for f in salarios_altos:
    print(f)


In [ ]:
# Funcionários que trabalham como 'Gerente'
cursor.execute("SELECT * FROM funcionarios WHERE cargo = 'Gerente';")
gerentes = cursor.fetchall()

print("Funcionários com cargo de Gerente:\n")
for f in gerentes:
    print(f)


### 2.4 — Atualização de dados (UPDATE)

Vamos aumentar o salário do funcionário **João** em **10%**. O cálculo é feito diretamente na cláusula `SET`, multiplicando o salário atual por `1.10`.


In [ ]:
cursor.execute(
    "UPDATE funcionarios SET salario = salario * 1.10 WHERE nome = 'João Silva';"
)
conn.commit()

# Conferindo o novo valor
cursor.execute("SELECT nome, salario FROM funcionarios WHERE nome = 'João Silva';")
print("Novo salário de João Silva:", cursor.fetchone())


### 2.5 — Exclusão de dados (DELETE)

Para exemplificar o `DELETE`, vamos remover o funcionário **Pedro Santos** da tabela.


In [ ]:
cursor.execute("DELETE FROM funcionarios WHERE nome = 'Pedro Santos';")
conn.commit()

print(f"✅ Registro(s) excluído(s): {cursor.rowcount}")

cursor.execute("SELECT * FROM funcionarios;")
print("\nTabela após a exclusão:")
for f in cursor.fetchall():
    print(f)


### 2.6 — Contagem total de registros

In [ ]:
cursor.execute("SELECT COUNT(*) FROM funcionarios;")
total = cursor.fetchone()[0]
print(f"Número total de funcionários cadastrados: {total}")


---
## 🧩 Parte 3 — Agregação e Ordenação

Nesta parte usamos **funções de agregação** (`SUM`, `AVG`, `MAX`, `MIN`), a cláusula **`GROUP BY`** para agrupar dados por cargo, e a cláusula **`ORDER BY`** para ordenar os resultados.


### 3.1 — Funções de agregação (SUM, AVG, MAX, MIN)

In [ ]:
cursor.execute("""
    SELECT
        SUM(salario) AS soma_total,
        AVG(salario) AS media_salarial,
        MAX(salario) AS maior_salario,
        MIN(salario) AS menor_salario
    FROM funcionarios;
""")
soma, media, maior, menor = cursor.fetchone()

print(f"💰 Soma total dos salários : R$ {soma:,.2f}")
print(f"📊 Salário médio          : R$ {media:,.2f}")
print(f"⬆️  Maior salário          : R$ {maior:,.2f}")
print(f"⬇️  Menor salário          : R$ {menor:,.2f}")


### 3.2 — Agrupamento de dados (GROUP BY)

In [ ]:
# Salário médio por cargo
cursor.execute("""
    SELECT cargo, AVG(salario) AS media_por_cargo
    FROM funcionarios
    GROUP BY cargo;
""")

print("Salário médio por cargo:\n")
for cargo, media in cursor.fetchall():
    print(f"{cargo:<15} R$ {media:,.2f}")


In [ ]:
# Quantidade de funcionários por cargo
cursor.execute("""
    SELECT cargo, COUNT(*) AS quantidade
    FROM funcionarios
    GROUP BY cargo;
""")

print("Quantidade de funcionários por cargo:\n")
for cargo, quantidade in cursor.fetchall():
    print(f"{cargo:<15} {quantidade} funcionário(s)")


### 3.3 — Ordenação de resultados (ORDER BY)

In [ ]:
# Funcionários ordenados do maior para o menor salário
cursor.execute("SELECT nome, salario FROM funcionarios ORDER BY salario DESC;")

print("Funcionários ordenados por salário (maior → menor):\n")
for nome, salario in cursor.fetchall():
    print(f"{nome:<20} R$ {salario:,.2f}")


In [ ]:
# Funcionários ordenados em ordem alfabética pelo nome
cursor.execute("SELECT nome, salario FROM funcionarios ORDER BY nome ASC;")

print("Funcionários em ordem alfabética:\n")
for nome, salario in cursor.fetchall():
    print(f"{nome:<20} R$ {salario:,.2f}")


---
## 🧩 Parte 4 — Transações e ROLLBACK

Uma **transação** é um conjunto de operações que deve ser executado de forma **atômica**: ou tudo é salvo (`COMMIT`), ou nada é salvo (`ROLLBACK`), garantindo a integridade do banco de dados.

Abaixo simulamos dois cenários:

1. Uma transação que **falha propositalmente** e é revertida com `conn.rollback()`.
2. Uma transação que **é concluída com sucesso** e é confirmada com `conn.commit()`.


### 4.1 — Simulação de transação com erro (ROLLBACK)

Vamos inserir um novo funcionário fictício, **Fernanda Costa**, e em seguida forçar um erro (uma divisão por zero) dentro do bloco `try`. Como o erro é capturado pelo `except`, chamamos `conn.rollback()` para desfazer a inserção — ou seja, Fernanda **não deve permanecer salva** no banco.


In [ ]:
try:
    cursor.execute(
        "INSERT INTO funcionarios (nome, cargo, salario, data_contratacao) VALUES (?, ?, ?, ?);",
        ("Fernanda Costa", "Estagiária", 1800.00, "2026-09-01")
    )

    # Forçando um erro proposital para acionar o "except"
    resultado = 10 / 0

    conn.commit()

except Exception as erro:
    print(f"❌ Erro detectado: {erro}")
    print("↩️  Revertendo a transação com conn.rollback()...")
    conn.rollback()

# Conferindo que Fernanda NÃO foi salva no banco
cursor.execute("SELECT * FROM funcionarios WHERE nome = 'Fernanda Costa';")
resultado_busca = cursor.fetchall()
print("\nRegistros encontrados para 'Fernanda Costa':", resultado_busca)
print("✅ Como esperado, a lista está vazia: a transação foi revertida com sucesso.")


### 4.2 — Transação concluída com sucesso (COMMIT)

Agora repetimos o processo, mas **sem forçar nenhum erro**. Inserimos o funcionário **Roberto Alves** e confirmamos a operação com `conn.commit()`, tornando-a permanente.


In [ ]:
try:
    cursor.execute(
        "INSERT INTO funcionarios (nome, cargo, salario, data_contratacao) VALUES (?, ?, ?, ?);",
        ("Roberto Alves", "Desenvolvedor", 6300.00, "2023-02-14")
    )

    # Nenhum erro é forçado desta vez
    conn.commit()
    print("✅ Transação concluída e salva com sucesso (commit realizado)!")

except Exception as erro:
    print(f"❌ Erro detectado: {erro}")
    conn.rollback()

# Verificando se Roberto agora faz parte da tabela
cursor.execute("SELECT * FROM funcionarios WHERE nome = 'Roberto Alves';")
print("\nRegistro de Roberto Alves:", cursor.fetchone())


---
## 🧩 Parte 5 — Joins e Múltiplas Tabelas

Nesta parte, o banco de dados passa a ter **múltiplas tabelas relacionadas**, simulando uma estrutura mais próxima da realidade: funcionários, departamentos e projetos.

Vamos:
1. Criar a tabela `departamentos`;
2. Adicionar a coluna `departamento_id` em `funcionarios` (chave estrangeira);
3. Relacionar cada funcionário a um departamento;
4. Fazer um `INNER JOIN` entre `funcionarios` e `departamentos`;
5. Criar as tabelas `projetos` e `funcionarios_projetos` (relação N:N);
6. Fazer um `LEFT JOIN` para listar todos os funcionários, com ou sem projeto associado.


### 5.1 — Criação da tabela `departamentos` e inserção de registros

In [ ]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS departamentos (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        nome_departamento TEXT NOT NULL,
        localizacao TEXT
    );
""")

departamentos_iniciais = [
    ("Tecnologia da Informação", "3º andar"),
    ("Recursos Humanos",         "1º andar"),
    ("Vendas",                   "2º andar"),
]

cursor.executemany(
    "INSERT INTO departamentos (nome_departamento, localizacao) VALUES (?, ?);",
    departamentos_iniciais
)
conn.commit()

cursor.execute("SELECT * FROM departamentos;")
print("Departamentos cadastrados:\n")
for d in cursor.fetchall():
    print(d)


### 5.2 — Adição da chave estrangeira `departamento_id` (ALTER TABLE)

O SQLite permite adicionar colunas a uma tabela já existente através de `ALTER TABLE ... ADD COLUMN`. Aqui adicionamos a coluna `departamento_id`, que passa a referenciar o `id` da tabela `departamentos`.

> ⚠️ O SQLite, por padrão, não aplica (enforce) restrições de chave estrangeira automaticamente — é preciso habilitar isso com `PRAGMA foreign_keys = ON`. Ainda assim, a coluna é declarada como `REFERENCES` por boa prática e clareza do modelo relacional.


In [ ]:
try:
    cursor.execute("ALTER TABLE funcionarios ADD COLUMN departamento_id INTEGER REFERENCES departamentos(id);")
    conn.commit()
    print("✅ Coluna 'departamento_id' adicionada com sucesso à tabela 'funcionarios'!")
except sqlite3.OperationalError as erro:
    # Evita erro caso o notebook seja executado mais de uma vez e a coluna já exista
    print(f"⚠️ Aviso: {erro} (a coluna provavelmente já existe)")

# Habilitando a checagem de chaves estrangeiras para esta conexão
cursor.execute("PRAGMA foreign_keys = ON;")

cursor.execute("PRAGMA table_info(funcionarios);")
print("\nEstrutura atualizada da tabela 'funcionarios':")
for coluna in cursor.fetchall():
    print(coluna)


### 5.3 — Associando cada funcionário a um departamento (UPDATE)

In [ ]:
# 1 = Tecnologia da Informação | 2 = Recursos Humanos | 3 = Vendas
associacoes = [
    (1, "João Silva"),
    (2, "Maria Souza"),
    (1, "Carlos Pereira"),
    (3, "Ana Lima"),
    (2, "Roberto Alves"),
]

for departamento_id, nome in associacoes:
    cursor.execute(
        "UPDATE funcionarios SET departamento_id = ? WHERE nome = ?;",
        (departamento_id, nome)
    )

conn.commit()

cursor.execute("SELECT nome, departamento_id FROM funcionarios;")
print("Funcionários e seus respectivos departamento_id:\n")
for f in cursor.fetchall():
    print(f)


### 5.4 — INNER JOIN: funcionários + departamentos

In [ ]:
cursor.execute("""
    SELECT funcionarios.nome, departamentos.nome_departamento
    FROM funcionarios
    INNER JOIN departamentos ON funcionarios.departamento_id = departamentos.id;
""")

print("Funcionários e seus departamentos (INNER JOIN):\n")
for nome, depto in cursor.fetchall():
    print(f"{nome:<20} → {depto}")


### 5.5 — Criação das tabelas `projetos` e `funcionarios_projetos` (relação N:N)

In [ ]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS projetos (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        nome_projeto TEXT NOT NULL
    );
""")

cursor.execute("""
    CREATE TABLE IF NOT EXISTS funcionarios_projetos (
        funcionario_id INTEGER,
        projeto_id INTEGER,
        FOREIGN KEY (funcionario_id) REFERENCES funcionarios(id),
        FOREIGN KEY (projeto_id) REFERENCES projetos(id)
    );
""")

projetos_iniciais = [
    ("Sistema ERP",),
    ("Aplicativo Mobile",),
    ("Site Institucional",),
]
cursor.executemany("INSERT INTO projetos (nome_projeto) VALUES (?);", projetos_iniciais)

# Relacionando apenas alguns funcionários a projetos, de propósito,
# para que o LEFT JOIN mais adiante mostre também quem NÃO está em nenhum projeto
relacoes_funcionario_projeto = [
    (1, 1),  # João Silva      -> Sistema ERP
    (3, 1),  # Carlos Pereira  -> Sistema ERP
    (3, 2),  # Carlos Pereira  -> Aplicativo Mobile
    (6, 3),  # Roberto Alves   -> Site Institucional (id 6, pois o id 5 foi usado por Pedro Santos, excluído na Parte 2)
    # Maria Souza (2) e Ana Lima (4) ficam de fora, propositalmente
]
cursor.executemany(
    "INSERT INTO funcionarios_projetos (funcionario_id, projeto_id) VALUES (?, ?);",
    relacoes_funcionario_projeto
)

conn.commit()
print("✅ Tabelas 'projetos' e 'funcionarios_projetos' criadas e populadas com sucesso!")


### 5.6 — LEFT JOIN: todos os funcionários, com ou sem projeto

O `LEFT JOIN` garante que **todos** os funcionários apareçam no resultado, mesmo aqueles que não estão vinculados a nenhum projeto — nesses casos, o nome do projeto aparece como `None` (equivalente ao `NULL` do SQL).


In [ ]:
cursor.execute("""
    SELECT funcionarios.nome, projetos.nome_projeto
    FROM funcionarios
    LEFT JOIN funcionarios_projetos ON funcionarios.id = funcionarios_projetos.funcionario_id
    LEFT JOIN projetos ON funcionarios_projetos.projeto_id = projetos.id
    ORDER BY funcionarios.nome;
""")

print("Funcionários e seus projetos (LEFT JOIN):\n")
for nome, projeto in cursor.fetchall():
    projeto_exibido = projeto if projeto else "— sem projeto associado —"
    print(f"{nome:<20} → {projeto_exibido}")


---
## 🧩 Parte 6 — DCL (Controle de Dados)

### 📚 DDL, DML, DQL e DCL — qual a diferença?

O SQL é dividido em sublinguagens de acordo com a **finalidade** de cada comando. As quatro principais são:

| Sigla | Nome completo | Finalidade | Exemplos de comandos |
|:-----:|----------------|------------|------------------------|
| **DDL** | *Data Definition Language* (Linguagem de Definição de Dados) | Define e altera a **estrutura** do banco de dados: tabelas, colunas, índices, relacionamentos. | `CREATE TABLE`, `ALTER TABLE`, `DROP TABLE` |
| **DML** | *Data Manipulation Language* (Linguagem de Manipulação de Dados) | Manipula os **dados armazenados** dentro das tabelas já existentes. | `INSERT`, `UPDATE`, `DELETE` |
| **DQL** | *Data Query Language* (Linguagem de Consulta de Dados) | Usada exclusivamente para **consultar** dados, sem alterá-los. Muitas vezes é tratada como parte da DML, mas conceitualmente é separada por seu propósito puramente de leitura. | `SELECT` |
| **DCL** | *Data Control Language* (Linguagem de Controle de Dados) | Controla o **acesso e as permissões** dos usuários sobre o banco de dados — quem pode ler, escrever, criar ou apagar objetos. | `GRANT`, `REVOKE` |

Neste notebook, usamos DDL nas Partes 1 e 5 (criação de tabelas e `ALTER TABLE`), DML nas Partes 2 e 4 (inserções, atualizações, exclusões e transações), e DQL em praticamente todas as consultas `SELECT` das Partes 2 e 3.

### 🔐 Para que serve o DCL?

O **DCL** é a parte da linguagem SQL responsável pela **segurança e governança do banco de dados**, definindo **quem pode fazer o quê**. Seus dois comandos principais são:

- **`GRANT`** → concede um privilégio específico a um usuário ou papel (*role*), como permissão para `SELECT`, `INSERT`, `UPDATE` ou até para administrar todo o banco.
- **`REVOKE`** → remove um privilégio anteriormente concedido.

Exemplo (sintaxe típica de SGBDs como PostgreSQL ou MySQL — o SQLite, por ser um banco de dados *serverless* e de arquivo único, **não implementa DCL**, pois não possui um sistema de usuários/permissões embutido):

```sql
-- Concede permissão de leitura e escrita ao usuário "analista_dados"
GRANT SELECT, INSERT, UPDATE ON funcionarios TO analista_dados;

-- Revoga a permissão de exclusão do mesmo usuário
REVOKE DELETE ON funcionarios FROM analista_dados;
```

Em resumo, enquanto DDL define **a estrutura**, DML e DQL cuidam **dos dados em si**, o DCL cuida de **quem tem permissão para acessar ou modificar essa estrutura e esses dados** — um pilar essencial para a segurança da informação em bancos de dados corporativos, com múltiplos usuários e níveis de acesso distintos (ex.: um estagiário pode ter permissão apenas de leitura, enquanto um DBA tem controle total).


---
## 🔚 Encerramento da Conexão

Por fim, seguindo a boa prática de sempre liberar os recursos utilizados, encerramos a conexão com o banco de dados.


In [ ]:
conn.close()
print("🔒 Conexão com o banco de dados 'empresa.db' encerrada com sucesso.")


---
## ✅ Considerações finais

Este notebook percorreu todo o ciclo básico de manipulação de um banco de dados relacional com Python e SQLite: desde a **criação da estrutura (DDL)**, passando pela **manipulação dos dados (DML)**, **consultas agregadas e ordenadas (DQL)**, o controle de **transações (COMMIT/ROLLBACK)**, o relacionamento entre **múltiplas tabelas (JOINs)**, até a explicação conceitual do **DCL**.

### 🧗 Dificuldades encontradas

*(espaço reservado para anotar, durante a aula, qualquer dificuldade enfrentada ao longo da resolução dos exercícios, conforme solicitado no enunciado)*

- ...
